#### 图像分类
- 最近邻
- 线性分类

计算机如何认识图像？

图像在计算机中表示为多维张量，具有 RGB 三个通道，每个通道的数值范围在 [0, 255] 之间。

传统方法通过边缘检测器寻找角点等特征，但这类方法泛化能力有限。

最近邻效果并不理想  
像素级比较对微小位移、光照变化非常敏感，且测试阶段计算量极大。

数据驱动方法  
1. 收集数据集和标签  
2. 用机器学习算法训练分类器  
3. 在新图像上测试分类器  

其中，训练集通常表示为：

$$
\{(x_i, y_i)\}_{i=1}^N
$$

其中 $x_i$ 表示第 $i$ 张输入图像，$y_i$ 表示其对应的真实标签，$N$ 为样本总数。

模型的目标是学习一个映射函数：

$$
f: X \to Y
$$

使得该函数能够将输入图像 $x$ 正确映射到其类别标签 $y$。

In [ ]:
import numpy as np
from typing import Any

def train(images: np.ndarray, labels: np.ndarray) -> Any:
    """
    Train a machine learning model.
    images: training images, shape (N, D)
    labels: training labels, shape (N,)
    Returns the trained model.
    """
    # Machine learning model training logic
    pass

def predict(model: Any, test_images: np.ndarray) -> np.ndarray:
    """
    Use the trained model to predict labels for test_images.
    model: trained model
    test_images: test images, shape (M, D)
    Returns predicted labels, shape (M,)
    """
    # Prediction logic
    pass

##### 最近邻分类器Nearest Neighbor

Distance Magic

距离计算是最近邻分类器的核心，用于衡量图片像素之间的相似度。

**L1距离**

L1距离又称曼哈顿距离，其几何特性是只能沿着坐标轴横着或竖着移动。

$$
d_1(I_1, I_2) = \sum_p |I_1^p - I_2^p|
$$

其中 $I_1$ 和 $I_2$ 代表两张图像，$p $为像素索引。

L1距离与L2距离的直观对比如下图所示：

![L1与L2距离对比图](../img/L1L2.png)

In [ ]:
import numpy as np
''' 
一个基于L1距离的最近邻图像分类器
它在训练阶段只是“死记硬背”所有图片和标签
在预测阶段则计算新图片与所有训练图片的像素差
找到最相似的那张图，直接把它的标签作为预测结果
'''

'''
输入：

X（图片数据）：是一个二维矩阵（Numpy Array），形状为 (N, D)。

N：图片的数量。比如训练集有50000张图。

D：每张图展平后的像素总数。32×32×3（RGB三通道） = 3072。

所以 X 就是一个 (50000, 3072) 的大矩阵，每一行代表一张被拉直成一条线的图片。

y（真实标签）：是一个一维数组，形状为 (N,)。

比如 y = [0, 5, 3, 2, ...]，里面的数字是 0 到 9，对应 10 个类别。

测试集输入：predict 里的 X 形状是 (M, D)，M 是测试图片数量（比如10000），D 依然是 3072。

'''

'''
输出：

X[i, :] 取出了第 i 张测试图片（一个长度为 D 的一维向量）。

self.Xtr - X[i, :] 利用了 NumPy 的广播机制，让所有训练图片（50000行）都减去这张测试图片。

np.abs(...) 取绝对值。

np.sum(..., axis=1) 把每行加起来，得到 50000 个距离值（L1距离）。

distances 是一个长度为 50000 的数组，代表这张测试图与所有训练图的像素差之和。

找出 distances 中最小值的索引，也就是“最像”的那张训练图的下标。

把最像的那张训练图的标签，直接作为第 i 张测试图的预测结果。

'''
class NearestNeighbor:
    def __init__(self):
        pass

    def train(self, X, y):
        # X is N x D where each row is an example.
        # Y is 1-dimensional of size N.
        # The nearest neighbor classifier simply remembers all the training data.
        self.Xtr = X
        self.ytr = y

    def predict(self, X):
        # X is N x D where each row is an example we wish to predict label for.
        num_test = X.shape[0]

        # Let's make sure that the output type matches the input type.
        Ypred = np.zeros(num_test, dtype=self.ytr.dtype)

        # Loop over all test rows.
        for i in range(num_test):
            # Find the nearest training image to the i'th test image
            # using the L1 distance (sum of absolute value differences).
            distances = np.sum(np.abs(self.Xtr - X[i, :]), axis=1)

            min_index = np.argmin(distances)  # Get the index with smallest distance.
            Ypred[i] = self.ytr[min_index]    # Predict the label of the nearest example.

        return Ypred


**L2距离**

L2距离又称欧氏距离，度量的是两点之间的直线距离。

数学公式如下：

$$
d_2(I_1, I_2) = \sqrt{\sum_p (I_1^p - I_2^p)^2}
$$

其中 $I_1$ 和 $I_2$ 代表两张图像，$p$ 为像素索引。

L1与L2的对比可参考前文的图示。

---

##### K-Nearest Neighbors

K近邻分类器是最近邻的推广。K=1时即为最近邻分类器，预测时取最近的1个训练样本的标签作为结果。K>1时，则选取距离最近的K个训练样本，通过投票决定最终预测类别。

[K-Nearst Neighbors 演示](http:vision.stanford.edu/teaching/cs231n-demos/knn/)

##### 超参数

K近邻算法中的K值和距离函数都是典型的超参数。超参数需要人为设置，不能由算法从数据中自动学习得到。

**K值的选择**

K值过小，模型对噪声敏感，容易过拟合。K=1时在训练集上永远能达到100%准确率，但这不代表模型泛化能力强。

K值过大，模型过于平滑，可能欠拟合。

**超参数选取策略**

常见的超参数选取策略有以下几种。

Idea #1：选择在训练集上表现最好的超参数。这是错误的，因为K=1永远在训练集上完美。

Idea #2：选择在测试集上表现最好的超参数。这也是错误的，这样会导致算法对测试集过拟合，实际部署时性能会远低于预期。

Idea #3：将数据分为训练集、验证集，在验证集上选择超参数，最后在测试集上评估。这是正确的做法。

Idea #4：交叉验证。将训练集分成若干份，轮流将其中一份作为验证集，其余作为训练集，最后取平均结果。适用于数据集较小的情况。

---

##### 数据集划分

**3. 训练集、验证集、测试集**

将可用数据划分为三部分：

- **训练集**：用于训练模型参数。
- **验证集**：用于调整超参数，选择最优配置。
- **测试集**：只在最后使用一次，用于评估最终模型的泛化性能。

以CIFAR-10为例，可以用49000张作为训练集，1000张作为验证集。

**Idea #3的具体流程**

在验证集上尝试不同的超参数，记录每个超参数对应的准确率，选择验证集上表现最好的超参数。然后用这个超参数在全部训练数据上重新训练，最后在测试集上跑一次，报告结果。

**4. 交叉验证**

当训练数据较少时，验证集数量也会很少，此时可以使用交叉验证。

将训练集平均分成 $k$ 份（通常k=3、5、10），每次用其中 $k-1$ 份训练，剩下1份验证，循环 $k$ 次，最后取 $k$ 次验证结果的平均值作为该超参数的性能估计。

这样做的好处是减少了验证集划分带来的噪声，得到更稳定的超参数选择。缺点是计算成本成倍增加。

如果训练数据充足，通常优先使用单次验证集划分，因为交叉验证计算开销较大。

##### 距离度量的局限性

K近邻使用像素级距离进行图像分类，实际效果并不理想。

原因有二：

第一，像素距离对微小的位移、旋转、光照变化非常敏感。同一物体经过平移或旋转后，像素值差异可能很大，导致被判定为不同类别。

第二，高维空间中的距离度量会失去区分度，这种现象被称为维数灾难。随着维度增加，所有点之间的距离趋于接近，距离度量不再具有信息量。

因此，K近邻搭配像素距离在图像分类中几乎不被实际使用。

---

##### 线性分类器 Linear Classifier

**映射**

线性分类器是一个把输入映射到输出的函数 $f(x, W)$，其中 $W$ 是权重矩阵。

输入图像大小为 $32 \times 32 \times 3$，展平后得到长度为 3072 的向量 $x$。函数 $f(x, W)$ 将其映射为 10 个类别的分数。

线性模型公式如下：

$$
f(x, W) = W x + b
$$

一张图像会对每个类别都输出一个分数。

线性分类器是神经网络的基础模块。

![f(x,W)](../img/f(x,W).png)

**损失函数与最大似然估计**

损失函数用于衡量预测分数与真实分数之间的差异。

基于最大似然估计，计算正确类别的概率，取对数，再取负值，就得到了损失。

**Softmax 公式**

Softmax 分类器把原始分数转换为概率分布。它本质上就是多分类逻辑回归。

Softmax 函数将分数 $s$ 映射为合为1概率：

$$
P(y = k | x) = \frac{e^{s_k}}{\sum_j e^{s_j}}
$$

负值概率趋于0。

对应的损失函数即交叉熵损失：

$$
L_i = -\log P(y_i | x_i)
$$

它衡量的是模型预测分布与真实分布之间的差异。


---

**图像识别时计算机会遇到很多挑战。**

![challenges](../img/COR.png)

#### 正则化和优化


##### 正则化 Regularization

核心思路：在训练集上表现稍差，但在未见过的数据上表现更好。倾向于选择拟合度稍低但更简单的模型。

通常不对偏置项进行正则化，因为偏置不控制特征的方向。

完整的损失函数由数据损失和正则化项组成：

$$
L = \frac{1}{N} \sum_i L_i + \lambda R(W)
$$

其中 $\lambda$ 是正则化强度，$R(W)$ 是正则化项。

**L2正则化**

对权重平方进行惩罚。对极小值的惩罚更小，倾向于让权重分散。

$$
R(W) = \sum_k \sum_l W_{k,l}^2
$$

**L1正则化**

对权重绝对值进行惩罚。倾向于产生稀疏的权重矩阵。

$$
R(W) = \sum_k \sum_l |W_{k,l}|
$$

**L1 + L2**

结合两者，又称 Elastic Net。

![Regularization](../img/Regularization.png)

##### 优化 Optimization

优化的目标是找到损失函数的最低点，即最优解。

核心方法是梯度下降。沿着梯度的反方向向下走，感受当前位置的损失，然后向下迈一步。Follow the slope。

**数值梯度与解析梯度**

数值梯度利用极限定义，取极小的 $h$ 来近似计算梯度。计算慢，但容易实现，常用于梯度检查。

$$
\frac{df}{dx} \approx \frac{f(x+h) - f(x-h)}{2h}
$$

解析梯度通过微积分推导得出，计算精确且快速，是反向传播使用的真正方法。

梯度检查用于验证解析梯度是否实现正确。

损失函数通常是可微的。对于凸函数，局部最小值就是全局最小值。

**梯度下降**

定好迭代次数，或者等待损失收敛。

##### 随机梯度下降 SGD

每次迭代只使用一小批数据来估计梯度。

SGD的问题在于：

1. 鞍点。梯度为0，容易卡住。
2. 噪声。子采样带来的梯度估计噪声，导致更新方向震荡。

##### SGD + Momentum

引入动量，对噪声进行平均，抑制震荡。

更新公式为：

$$
v = \rho v - \alpha \nabla L
$$

$$
x = x + v
$$

其中 $\rho$ 是动量系数，通常取 0.9 左右。$\alpha$ 是学习率。

动量能让收敛可能更慢，但容易找到更优的极小值。因为积累了历史速度，可能会在最小值附近超调，但通过后续调整最终能稳定下来。

In [ ]:
# gradient descent

def compute_gradient(x ,batch_data):
    # Compute the gradient of the loss function with respect to x
    # This is a placeholder function; replace with actual gradient computation
    return np.random.randn(*x.shape)  # Example: random gradient for demonstration


# SGD + Momentum

vx = 0  # Initialize velocity as zero vector
while True:
    dx = compute_gradient(x, batch_data)
    # Placeholder: returns a random array with the same shape as x
    # In practice, this should be the real gradient (e.g., 2*x for f(x)=x^2)

    vx = rho * vx + dx  # v = rho * v + gradient
    x -= learning_rate * vx



##### RMSProp

RMSProp 的核心思路是：**少往陡峭的地方走，多往平坦的地方走。**

它通过计算梯度平方的指数衰减平均，来自适应地调整每个参数的学习率。

更新公式为：

$$
s = \beta s + (1 - \beta) (\nabla L)^2
$$

$$
x = x - \alpha \frac{\nabla L}{\sqrt{s} + \epsilon}
$$

其中 $\beta$ 是衰减率，通常取 0.9 或 0.99。

当梯度大（陡峭）时，$s$ 大，分母大，实际步长变小。当梯度小（平坦）时，$s$ 小，分母小，实际步长变大。

##### Adam (almost)

Adam 本质上是 RMSProp 和 Momentum 的结合。

它同时计算梯度的一阶矩估计（动量）和二阶矩估计（梯度平方），并引入**偏差校正**。

偏差校正解决了初始步长过大的问题。因为在训练初期，$m$ 和 $v$ 初始化接近 0，如果不校正，更新量会非常小。校正后，早期更新步长被放大，模型能快速启动。

Adam 中动量计算时的梯度只看数据损失，最后加上正则化。完整的梯度为：

$$
g_t = \nabla f(w_t) + \lambda w_t
$$

其中 $\lambda w_t$ 是正则化项。

更新公式为：

$$
m = \beta_1 m + (1 - \beta_1) g_t
$$

$$
v = \beta_2 v + (1 - \beta_2) g_t^2
$$

$$
\hat{m} = \frac{m}{1 - \beta_1^t}, \quad \hat{v} = \frac{v}{1 - \beta_2^t}
$$

$$
x = x - \alpha \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon}
$$

##### AdamW

标准的 Adam 在处理正则化时存在缺陷。因为 L2 正则化被加进了梯度里，它会和自适应学习率交互，导致正则化效果被削弱。

AdamW 的核心是**解耦权重衰减（Decoupled Weight Decay）**。它将权重衰减从梯度更新中分离出来，直接作用于权重本身。

更新公式为：

$$
x = x - \alpha \left( \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon} + \lambda x \right)
$$

**关于“跑固定轮次后，学习率除以10”**：这不是 AdamW 的专利，而是**步长衰减（StepLR）**。这是一种通用的学习率调度策略，通常每跑固定轮次后，学习率乘以 0.1。它和 AdamW 是正交的概念，可以搭配使用。

##### 余弦学习率衰减

学习率遵循余弦曲线，从初始值平滑衰减到 0 或一个小值。

$$
\alpha_t = \frac{1}{2} \alpha_0 \left( 1 + \cos\left( \frac{t \pi}{T} \right) \right)
$$

其中 $T$ 是总轮次。相比 StepLR，余弦衰减更平滑，后期学习率极小，有助于模型精细收敛。

##### 线性预热

在训练刚开始时，模型权重是随机的，梯度可能很大。如果直接用大学习率，容易导致训练不稳定。

线性预热在最初的几个轮次里，将学习率从 0 或一个极小值线性增加到初始学习率。

```python
# Linear Warmup (pseudocode)
if current_step < warmup_steps:
    lr = base_lr * current_step / warmup_steps
else:
    lr = base_lr
```

##### 线性缩放定律

这是一个经验法则。当批量大小乘以 $k$ 时，学习率也应该乘以 $k$（或 $\sqrt{k}$）。这有助于在大批量训练时保持梯度更新的方差大致恒定。

##### 海森矩阵与大模型

海森矩阵是二阶导数矩阵，能提供损失函数的曲率信息，理论上能帮助优化器找到更好的下降方向。

但在大模型中基本不用。因为参数量巨大，海森矩阵的维度是平方级别，计算和存储成本极高。即使是对角近似，也极其昂贵。因此，大模型训练几乎全部依赖 Adam、AdamW 等一阶优化方法。

In [ ]:
import numpy as np

# Adam Optimizer

first_moment = 0        # m: first moment estimate 
#first_moment = np.zeros_like(x)

second_moment = 0       # v: second moment estimate

beta1 = 0.9             # Decay rate for first moment
beta2 = 0.999           # Decay rate for second moment
learning_rate = 0.001
epsilon = 1e-8

for t in range(1, num_iterations + 1):
    dx = compute_gradient(x)    # g_t: gradient at current step
    
    # Update biased first moment estimate (m = beta1 * m + (1 - beta1) * g)
    first_moment = beta1 * first_moment + (1 - beta1) * dx
    
    # Update biased second raw moment estimate (v = beta2 * v + (1 - beta2) * g^2)
    second_moment = beta2 * second_moment + (1 - beta2) * (dx ** 2)
    
    # Compute bias-corrected first moment estimate
    first_moment_corrected = first_moment / (1 - beta1 ** t)
    
    # Compute bias-corrected second raw moment estimate
    second_moment_corrected = second_moment / (1 - beta2 ** t)
    
    # Update parameters (x = x - lr * m_hat / (sqrt(v_hat) + epsilon))
    x -= learning_rate * first_moment_corrected / (np.sqrt(second_moment_corrected) + epsilon)


In [ ]:
# AdamW (Decoupled Weight Decay) 解耦权重衰减,修改最后一行
x -= learning_rate * (first_moment_corrected / (np.sqrt(second_moment_corrected) + epsilon) + weight_decay * x)

##### SVM损失函数

**用途**

SVM损失是数据损失的核心组成部分，用于衡量模型输出的预测分数与真实标签之间的差距。它通常用于线性分类器或神经网络的输出层，为模型提供优化方向。

**核心原理：安全边界与合页损失**

SVM的核心思想是：不仅要预测正确，还要自信地预测正确。它希望正确类别的分数，比其他所有错误类别的分数，至少高出一个安全边界，即 Margin，通常设为 $\Delta = 1.0$。

如果正确类别的分数比某个错误类别的分数高出至少 $\Delta$，模型就认为在这个类别上已经足够安全，不再产生损失，即 Loss = 0。否则，就会产生惩罚，即 Loss > 0。这种机制被称为合页损失 Hinge Loss。

**数学公式**

对于第 $i$ 个样本，多类SVM损失的公式为：

$$
L_i = \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + \Delta)
$$

其中 $s_j$ 是模型对第 $j$ 个错误类别的预测分数，$s_{y_i}$ 是模型对真实类别 $y_i$ 的预测分数，$\Delta$ 是安全边界，通常取 1.0。$\max(0, \cdot)$ 就是合页损失，如果括号内小于0，即已经足够安全，则损失为0。

整个数据集的平均SVM损失为：

$$
L = \frac{1}{N} \sum_{i=1}^N L_i + \lambda R(W)
$$

其中 $\lambda R(W)$ 是正则化项。

**直观例子**

假设有3个类别，分别是猫、狗、汽车，真实标签是猫，即 $y_i = 0$。模型的预测分数为：猫 3.2，狗 5.1，汽车 -1.7。设边界 $\Delta = 1.0$。

对于错误类别狗，$\max(0, 5.1 - 3.2 + 1.0) = \max(0, 2.9) = 2.9$，产生损失。

对于错误类别汽车，$\max(0, -1.7 - 3.2 + 1.0) = \max(0, -3.9) = 0$，不产生损失。

所以这个样本的损失为 2.9 + 0 = 2.9。

这说明模型虽然预测对了猫，但只比狗高了1.9分，没有达到安全边界1.0的自信差距，因此被罚款。

**SVM损失的作用**

指导优化方向。它为模型提供了一个可微的梯度信号，驱动模型去拉大正确类别与错误类别之间的分数差距。

控制模型行为。边界 $\Delta$ 决定了模型要多自信才算满意。$\Delta$ 越大，模型被迫拉开的分数差距就越大。

与Softmax的对比。SVM只关心分数是否超过边界，不关心分数之间的绝对差异。而Softmax，也就是交叉熵，则会把分数转化为概率分布，关心正确类别的概率有多大。这是两种不同的优化哲学。

---

#### 神经网络与反向传播

##### SVM损失函数
SVM损失函数通常用于线性分类器，用来衡量预测分数与真实分数之间的差异。它属于数据损失的一部分。

##### 线性映射
线性分类器的基础公式是：

$$
f = Wx
$$

这种模型只能解决线性可分的问题。

##### 两层神经网络
在线性模型的基础上引入隐藏层，得到两层神经网络：

$$
f = W_2 \max(0, W_1 x)
$$

其中 $W_1$ 是第一层权重，$W_2$ 是第二层权重。$\max(0, \cdot)$ 即 ReLU 激活函数。完整形式通常会加上偏置项 $b_1$ 和 $b_2$：

$$
f = W_2 \max(0, W_1 x + b_1) + b_2
$$

##### 激活函数
激活函数引入非线性机制。如果没有激活函数，多层神经网络无论叠多深，本质上仍然等价于一个线性变换。

**ReLU**

ReLU 即整流线性单元。公式为：

$$
f(x) = \max(0, x)
$$

计算简单，收敛速度快，是当前最常用的默认激活函数。

**死神经元**

当某个神经元的权重使得其对所有输入都输出负数时，ReLU 的梯度为 0。该神经元将永久失活，不再更新，这叫死神经元问题。

**Leaky ReLU**

为了解决死神经元问题，Leaky ReLU 在负半轴引入一个小斜率 $\alpha$：

$$
f(x) = \max(\alpha x, x)
$$

通常 $\alpha$ 取 0.01 左右。

**ELU**

ELU 即指数线性单元，在负半轴使用指数函数：

$$
f(x) = \begin{cases} x & x > 0 \\ \alpha(e^x - 1) & x \le 0 \end{cases}
$$

负半轴均值接近 0，有助于加速收敛，但计算量稍大。

**etc...**

还有其他变体，例如 GELU、Swish、Maxout 等。

##### 制造非线性
激活函数的作用就是制造非线性。没有非线性，再深的网络也只是线性模型。

##### 全连接神经网络
全连接神经网络也叫多层感知机 MLP。每一层的每个神经元都与前一层的所有神经元相连。通过堆叠多个全连接层并配合激活函数，网络可以拟合极其复杂的非线性函数。

例如，在一个简单的线性分类器中，前向传播可能仅仅是：

$$
f = W x + b
$$

而在一个简单的两层神经网络中，前向传播引入了激活函数，可能如下：

$$
f = W_2 \max(0, W_1 x + b_1) + b_2
$$



In [ ]:
# forward pass 
import numpy as np

def forward_pass(x, W1, b1, W2, b2):
    """
    Simple forward pass for a 2-layer neural network.
    x: input data, shape (N, D)
    W1, b1: weights and bias of the first layer
    W2, b2: weights and bias of the second layer
    Returns the output scores.
    """
    # First layer: linear transformation followed by ReLU activation
    hidden = np.maximum(0, x @ W1 + b1)
    
    # Second layer: linear transformation to output scores
    scores = hidden @ W2 + b2
    
    return scores

In [ ]:
# a 2-layer neural network 

import numpy as np

# Define network dimensions
N, D_in, H, D_out = 64, 1000, 100, 10

# Randomly initialize input data, target, and weights
x = np.random.randn(N, D_in)
y = np.random.randn(N, D_out)
w1 = np.random.randn(D_in, H)
w2 = np.random.randn(H, D_out)

learning_rate = 1e-4

for t in range(500):
    # Forward pass: compute predicted y
    # First layer: linear transformation followed by Sigmoid activation
    h = 1 / (1 + np.exp(-x.dot(w1)))
    # Second layer: linear transformation to output
    y_pred = h.dot(w2)

    # Compute loss using squared error
    loss = np.square(y_pred - y).sum()
    if t % 100 == 0:
        print(f"Iteration {t}, Loss: {loss}")

    # Backward pass: compute gradients
    # Gradient of loss with respect to y_pred
    grad_y_pred = 2.0 * (y_pred - y)
    # Gradient of loss with respect to w2
    grad_w2 = h.T.dot(grad_y_pred)
    # Gradient of loss with respect to hidden layer output h
    grad_h = grad_y_pred.dot(w2.T)
    # Add Sigmoid derivative: multiply by h * (1 - h)
    grad_h = grad_h * h * (1 - h)
    # Gradient of loss with respect to w1
    grad_w1 = x.T.dot(grad_h)

    # Update weights using gradient descent
    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2


##### 正则化与网络规模的区别

**正则化**用于限制模型权重的大小，防止过拟合，比如 **L1 和 L2 正则化**。**网络规模**指层数和神经元数量，属于模型架构设计。不能用**缩小网络规模**来替代正则化。缩小网络会直接降低模型的表示能力，容易导致**欠拟合**，而正则化是在保持模型容量的前提下约束参数。

##### 神经元与激活函数

神经网络中的每个神经元可以视为计算图中的一个**门单元**。**激活函数**是一类特殊的门，引入**非线性**。前向传播时，它将输入映射为输出。反向传播时，它根据输入和输出计算**局部梯度**，并将**上游梯度**传递给**下游梯度**。

##### 计算图

![计算图](../img/backpropagation.png)

**计算图**将复杂函数拆解为一系列简单的中间步骤。**前向传播**时，依次计算中间节点的输出并保存。**反向传播**时，利用**链式法则**从输出层向输入层逐级求导。

##### 反向传播的目标与流程

反向传播的目的是计算**损失 $L$** 对所有变量包括**权重 $W$** 和**偏置 $b$** 的梯度。这些梯度将被**优化器**用来更新权重。

在实际框架中，不会为每一层手动写出导数函数，而是通过逐级反向传播自动完成梯度计算。

反向传播的完整流程如下。

1. **前向传播**求出每一步的中间输入和输出，保存在内存中供后续求导使用。
2. **反向传播**从末端开始。**末端梯度**通常恒为 1，即 $\frac{\partial L}{\partial L} = 1$。
3. 每一步先求**局部梯度**，再乘以**上游梯度**，得到**下游梯度**，并将其继续向下游传递。

##### 门单元及其梯度传播规则

**加法门**是**梯度分配器**。对于 $z = x + y$，局部梯度 $\frac{\partial z}{\partial x} = 1$，$\frac{\partial z}{\partial y} = 1$。加法门将上游梯度原封不动地分发给两个输入。

**乘法门**是**梯度交换器**。对于 $z = x \cdot y$，局部梯度 $\frac{\partial z}{\partial x} = y$，$\frac{\partial z}{\partial y} = x$。乘法门把上游梯度乘以另一个输入的值后再分发。

**复制门**用于把一个变量复制到多个分支。反向传播时，来自不同分支的梯度会在复制点进行累加，因为同一个变量对多个输出都有贡献。

**最大值门**是**梯度路由器**。对于 $z = \max(x, y)$，梯度只会传递给前向传播中数值较大的那个输入，另一个输入的梯度为 0。

##### Sigmoid 门

**Sigmoid 函数**作为激活函数时，其局部梯度为 $\sigma'(z) = \sigma(z)(1 - \sigma(z))$。反向传播时，上游梯度乘以这个局部梯度得到下游梯度。由于 Sigmoid 的导数最大值仅为 0.25，在深层网络中使用会导致梯度逐层衰减，产生**梯度消失问题**。


In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 前向传播
def forward_pass(w0, x0, w1, x1, w2):
    s0 = w0 * x0        # 乘法门
    s1 = w1 * x1        # 乘法门
    s2 = s0 + s1        # 加法门
    s3 = s2 + w2        # 加法门
    L = sigmoid(s3)     # Sigmoid 门
    return s0, s1, s2, s3, L

# 前向传播计算
w0, x0, w1, x1, w2 = 2.0, 3.0, 1.0, 4.0, 0.5
s0, s1, s2, s3, L = forward_pass(w0, x0, w1, x1, w2)

# 反向传播
grad_L = 1.0            # 末端梯度恒为 1，即 dL/dL
grad_s3 = grad_L * (L * (1 - L))    # Sigmoid 门：乘以局部梯度 L*(1-L)

grad_s2 = grad_s3 * 1.0             # 加法门：梯度原样分发
grad_w2 = grad_s3 * 1.0             # 加法门：梯度原样分发

grad_s0 = grad_s2 * 1.0             # 加法门：梯度原样分发
grad_s1 = grad_s2 * 1.0             # 加法门：梯度原样分发

grad_w0 = grad_s0 * x0              # 乘法门：乘以另一个输入 x0
grad_x0 = grad_s0 * w0              # 乘法门：乘以另一个输入 w0

grad_w1 = grad_s1 * x1              # 乘法门：乘以另一个输入 x1
grad_x1 = grad_s1 * w1              # 乘法门：乘以另一个输入 w1

print(f"L: {L}")
print(f"grad_w0: {grad_w0}, grad_x0: {grad_x0}")
print(f"grad_w1: {grad_w1}, grad_x1: {grad_x1}")
print(f"grad_w2: {grad_w2}")

In [ ]:
import torch

# 前向传播和反向传播的接口
class Multiply(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input1, input2):
        # ctx 是上下文对象，用于在前向和反向传播之间共享数据
        # 保存输入张量，反向传播时需要用它们计算局部梯度
        ctx.save_for_backward(input1, input2)
        # 前向计算：返回两个输入的乘积
        return input1 * input2

    @staticmethod
    def backward(ctx, grad_output):
        # grad_output 是损失函数对 forward 输出结果的梯度（上游梯度）
        # 从上下文中取出前向传播时保存的输入张量
        input1, input2 = ctx.saved_tensors
        # 乘法门的局部梯度：对 input1 的梯度等于 grad_output 乘以 input2
        grad_input1 = grad_output * input2
        # 对 input2 的梯度等于 grad_output 乘以 input1
        grad_input2 = grad_output * input1
        # 返回损失对两个输入的梯度，顺序必须与 forward 的输入顺序一致
        return grad_input1, grad_input2



vector to vector

**向量对向量求导**

当函数输入是向量，输出也是向量时，对输出向量 $\mathbf{y}$ 的每个元素关于输入向量 $\mathbf{x}$ 的每个元素求偏导，得到雅可比矩阵。

$$
\mathbf{J} = \frac{\partial \mathbf{y}}{\partial \mathbf{x}} = 
\begin{bmatrix}
\frac{\partial y_1}{\partial x_1} & \cdots & \frac{\partial y_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial y_m}{\partial x_1} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{bmatrix}
$$

**损失函数 $L$ 是标量**

当最终损失 $L$ 是一个标量时，$L$ 对向量或矩阵求导的结果称为梯度，其形状与被求导的变量完全相同。比如 $L$ 对矩阵 $W$ 的梯度 $\frac{\partial L}{\partial W}$，其形状与 $W$ 一致。

**矩阵乘法的反向传播**

对于矩阵乘法 $y = xW$，输入 $x$ 是 $N \times D$ 的矩阵，权重 $W$ 是 $D \times M$ 的矩阵，输出 $y$ 是 $N \times M$ 的矩阵。

在反向传播时，不需要构造巨大的雅可比矩阵。$x$ 的第 $i$ 行只影响 $y$ 的第 $i$ 行。上游梯度 $\frac{\partial L}{\partial y}$ 的形状是 $N \times M$。根据链式法则，可以通过矩阵乘法直接求出对 $x$ 和 $W$ 的梯度：

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} W^T
$$

$$
\frac{\partial L}{\partial W} = x^T \frac{\partial L}{\partial y}
$$

为矩阵乘法写反向传播函数，利用矩阵乘法计算梯度，避免了逐个元素计算雅可比矩阵。

```python
import numpy as np

class MatMul:
    def __init__(self):
        self.x = None
        self.W = None

    def forward(self, x, W):
        # 保存前向传播的输入，反向传播计算局部梯度时需要用到
        self.x = x
        self.W = W
        # 前向计算：矩阵乘法
        return x.dot(W)

    def backward(self, dout):
        # dout 是上游传下来的梯度，形状为 (N, M)
        # 损失对输入 x 的梯度，形状为 (N, D)
        dx = dout.dot(self.W.T)
        # 损失对权重 W 的梯度，形状为 (D, M)
        dW = self.x.T.dot(dout)
        return dx, dW
```